# H2 Phase 2 — Synthetic Pretraining

**Accelerator:** GPU T4 x1

V3 CRNN modelini 300K synthetic image üzerinde 8 epoch pretrain eder.

**Tahmini süre:** ~2 saat (T4)

**Önkoşul:**
- Phase 1'in `synthetic_data.zip` çıktısını Kaggle Dataset olarak ekle:
  `Add Data → Your Datasets → synthetic_data`

**Çıktı:** `pretrain_best.pth` → Phase 3 notebook'una dataset olarak ekleyeceksin.

---
**Donanım (makale için):**
- GPU: NVIDIA Tesla T4, 15360 MiB VRAM
- CPU: Intel(R) Xeon(R) CPU @ 2.20GHz
- RAM: 13 GB
- Env: Kaggle Notebooks, Python 3.10, PyTorch 2.3.0+cu121

In [ ]:
# Hücre 1: GPU kontrolü
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# CPU bilgisi (makale için)
!cat /proc/cpuinfo | grep 'model name' | head -1
!free -h | grep Mem

In [ ]:
# Hücre 2: crnn-h2-code dataset'ten kopyala + bağımlılıklar
import sys, os, shutil

CODE_INPUT = "/kaggle/input/datasets/brht25/crnn-h2-code"

if os.path.exists(CODE_INPUT):
    os.makedirs("/kaggle/working/cloud", exist_ok=True)
    for fname in os.listdir(f"{CODE_INPUT}/cloud"):
        if fname.endswith((".py", ".txt", ".sh")):
            shutil.copy(f"{CODE_INPUT}/cloud/{fname}", f"/kaggle/working/cloud/{fname}")
    shutil.copy(f"{CODE_INPUT}/trigram_lm.py", "/kaggle/working/trigram_lm.py")
    print("Scripts kopyalandı OK")
else:
    print(f"⚠️  {CODE_INPUT} bulunamadı — Add Data → Your Datasets → crnn-h2-code")

sys.path.insert(0, "/kaggle/working")
os.chdir("/kaggle/working")
!pip install -q -r cloud/requirements.txt

In [ ]:
# Hücre 3: Synthetic data path'ini bul
# Phase 1 çıktısı dataset olarak eklendiyse /kaggle/input/ altında olur
import os, glob

# Olası yerler
candidates = [
    "/kaggle/input/synthetic-data/synthetic_data",
    "/kaggle/input/h2-phase1-output/synthetic_data",
    "/kaggle/working/synthetic_data",
]

SYNTH_DIR = None
for c in candidates:
    if os.path.exists(c + "/labels.txt"):
        SYNTH_DIR = c
        break

if SYNTH_DIR:
    n = len(glob.glob(f"{SYNTH_DIR}/words/*.png"))
    print(f"Synthetic data bulundu: {SYNTH_DIR} ({n:,} images)")
else:
    print("\n⚠️ Synthetic data bulunamadı!")
    print("Mevcut input datasets:")
    for d in os.listdir("/kaggle/input"):
        print(f"  /kaggle/input/{d}")
    print("\nYukarıdaki listeden doğru path'i SYNTH_DIR'e yaz ve bu hücreyi tekrar çalıştır.")
    SYNTH_DIR = "/kaggle/input/BURAYA_YAZ/synthetic_data"

In [ ]:
# Hücre 4: Pretraining
CKPT_DIR = "/kaggle/working/checkpoints"

!python cloud/phase2_pretrain.py \
    --epochs 8 \
    --batch 128 \
    --lr 1e-3 \
    --synthetic-dir {SYNTH_DIR} \
    --ckpt-dir {CKPT_DIR}

In [ ]:
# Hücre 5: Sonuç özeti + checkpoint doğrulama
import json, os

hist_path = f"{CKPT_DIR}/pretrain_history.json"
ckpt_path = f"{CKPT_DIR}/pretrain_best.pth"

if os.path.exists(hist_path):
    with open(hist_path) as f:
        h = json.load(f)
    print(f"Epoch sonuçları:")
    for i, (wa, cer) in enumerate(zip(h['val_wa'], h['val_cer']), 1):
        print(f"  Epoch {i}: val_WA={wa*100:.2f}%  val_CER={cer*100:.2f}%")
    best_wa = max(h['val_wa'])
    print(f"\nBest synthetic val WA: {best_wa*100:.2f}%")
    if best_wa < 0.85:
        print("⚠️ %85 altında — fontlar yüklü mü, augmentation çok agresif mi?")
    else:
        print("✓ Beklenen aralıkta (>%85)")

if os.path.exists(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / 1024**2
    print(f"\nCheckpoint: {ckpt_path} ({size_mb:.1f} MB)")
    print("Bu dosyayı indirip Phase 3 notebook'una dataset olarak ekle.")
else:
    print("⚠️ pretrain_best.pth bulunamadı!")